# PINN chiller pipeline (P100)

Thin driver: clones the repo, attaches the private chiller dataset (and the
artifact cache when present), restores only protocol-compatible checkpoints,
runs train-only PySR + validation-only lambda selection + final test evaluation,
and packages `results/` as a downloadable tarball.
All experiment logic lives in the repo (`src/`), not in this notebook.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = 'https://github.com/kushc2004/pinn-dev.git'
repo = Path('/kaggle/working/pinn-dev')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo)], check=True)
subprocess.run(['pip', 'install', '--quiet', 'pysr==1.5.10'], check=True)

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

matches = list(Path('/kaggle/input').rglob('cleaned_chiller-3.csv'))
if len(matches) != 1:
    raise RuntimeError(f'Expected exactly one attached chiller dataset, found: {matches}')
data_dir = repo / 'data'
data_dir.mkdir(exist_ok=True)
shutil.copy(matches[0], data_dir / 'cleaned_chiller-3.csv')
print(f'Data staged from {matches[0]}', flush=True)

artifact_archives = list(Path('/kaggle/input').glob('*/pinn_artifacts.tar.gz'))
if artifact_archives:
    subprocess.run(
        ['python', 'scripts/restore_kaggle_artifacts.py', '--archive', str(artifact_archives[0])],
        cwd=repo, check=True,
    )
else:
    print('No artifact cache attached; running from scratch')

In [ ]:
subprocess.run(['python', '-m', 'compileall', '-q', 'src', 'scripts'], cwd=repo, check=True)
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=repo, check=True)
subprocess.run(['sh', 'scripts/run_full_experiment.sh'], cwd=repo, check=True)

In [ ]:
subprocess.run(['python', 'scripts/publish_kaggle_artifacts.py', '--no-upload'], cwd=repo, check=True)
shutil.copy(repo / 'artifacts/kaggle/pinn_artifacts.tar.gz', '/kaggle/working/pinn_artifacts.tar.gz')
print('Saved /kaggle/working/pinn_artifacts.tar.gz', flush=True)